<a href="https://colab.research.google.com/github/gracenaomi1122/my-first-repo/blob/main/Assignment_Classification_Metrics_and_Performance_Evaluation_Subjective.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import io
import matplotlib
matplotlib.use('Agg')  # Prevents GUI issues
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_curve,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score
)

# -----------------------------
# Step 1: Load the sample data
# -----------------------------
data_str = """V1,V2,V3,Class
-1.36,0.87,0.23,1
1.19,0.26,0.17,0
0.13,-0.17,-0.45,0
-0.97,1.79,0.86,0
-0.64,-0.34,0.46,1
1.49,1.21,-0.33,0
-0.04,-0.26,0.07,0
0.40,0.96,0.53,0
-0.15,-0.61,-0.23,1
0.22,0.53,-0.39,0
-1.11,0.08,0.74,1
0.65,-0.97,-0.61,0
"""

df = pd.read_csv(io.StringIO(data_str))

# -----------------------------
# Step 2: Separate features & target
# -----------------------------
X = df.drop("Class", axis=1)
y = df["Class"]

# -----------------------------
# Step 3: Train-test split
# -----------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# -----------------------------
# Step 4: Scale features
# -----------------------------
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# -----------------------------
# Step 5: Train Logistic Regression
# -----------------------------
model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train_scaled, y_train)

# -----------------------------
# Step 6: Predict probabilities
# -----------------------------
y_proba = model.predict_proba(X_test_scaled)[:, 1]

# -----------------------------
# Step 7: ROC Curve & AUC
# -----------------------------
fpr, tpr, thresholds = roc_curve(y_test, y_proba)
auc_score = roc_auc_score(y_test, y_proba)

print(f"AUC Score: {auc_score:.4f}")

plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, label=f"AUC = {auc_score:.4f}")
plt.plot([0, 1], [0, 1], linestyle="--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend(loc="lower right")
plt.grid(True)

# Save the figure
plt.savefig("roc_curve.png")
plt.close()

# -----------------------------
# Step 8: Threshold Evaluation
# -----------------------------
print("\nThreshold Metrics")
print("-" * 45)
print("{:<10} {:<10} {:<10} {:<10}".format(
    "Threshold", "Precision", "Recall", "F1 Score"
))

thresholds_to_try = [0.1, 0.2, 0.3, 0.4, 0.5]

for t in thresholds_to_try:
    y_pred = (y_proba >= t).astype(int)

    precision = precision_score(y_test, y_pred, zero_division=0)
    recall = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)

    print("{:<10.1f} {:<10.4f} {:<10.4f} {:<10.4f}".format(
        t, precision, recall, f1
    ))

# -----------------------------
# Step 9: Recommendation
# -----------------------------
print("\nRecommendation:")
print(
    "Threshold = 0.1 is recommended because it achieves the highest Recall, "
    "which minimizes missed fraud cases. Although it may increase false "
    "positives, this aligns with the company policy of prioritizing fraud "
    "detection over incorrectly flagging some legitimate transactions."
)

print("\nROC curve has been saved as 'roc_curve.png'.")

AUC Score: 1.0000

Threshold Metrics
---------------------------------------------
Threshold  Precision  Recall     F1 Score  
0.1        1.0000     1.0000     1.0000    
0.2        1.0000     1.0000     1.0000    
0.3        1.0000     1.0000     1.0000    
0.4        0.0000     0.0000     0.0000    
0.5        0.0000     0.0000     0.0000    

Recommendation:
Threshold = 0.1 is recommended because it achieves the highest Recall, which minimizes missed fraud cases. Although it may increase false positives, this aligns with the company policy of prioritizing fraud detection over incorrectly flagging some legitimate transactions.

ROC curve has been saved as 'roc_curve.png'.
